# M0 — ViZDoom smoke test on Kaggle

Settings: **Accelerator: GPU T4 x2**, **Internet: on**. Run with *Save & Run All* (commit mode).

The M0 gate itself is CPU-only. It also passes in a CPU-only session, which costs no GPU quota. Run it once on a GPU session as well to confirm the image that training will use.

If the repo is private, add a Kaggle Secret named `GITHUB_TOKEN`.

In [ ]:
import os
# Must be set before anything imports torch or vizdoom (PLAN: Kaggle operating rules 6).
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

REPO_URL = "https://github.com/Satyajit-147/VizDoom.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/rl-qa-tester"

In [ ]:
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repo, or no secret attached

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e ".[train,dev]"

In [ ]:
import sys, platform, vizdoom as vzd, torch
print("python ", sys.version.split()[0], platform.platform())
print("vizdoom", vzd.__version__, "| notifications buffer:", hasattr(vzd.DoomGame(), "set_notifications_buffer_enabled"))
print("torch  ", torch.__version__, "| cuda:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
print("cpus   ", os.cpu_count())

In [ ]:
!python -m pytest tests/test_smoke.py tests/test_no_groundtruth_leak.py -q -s